# Overview of measurements

| | |
|---|---|
| **Workflow** | optional, after step 2 |
| **Input** | pastastore per transect |
| **Output** | overview table, figures in the transect folder 'timeseries' |

An overview of all time series of all transects: period, number of values, time step and range, with the series, their data availability and a map per transect. Use it to check the data and to choose the model period (`tmin`, `tmax` in the transects metadata).

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import pandas as pd

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]

## 1. Overview per transect

In [ ]:
rows = []
for transect in transects:
    pstore = project.open_pastastore(transect)
    names = pstore.oseries_names
    if not names:
        print(f"{transect}: the pastastore is empty")
        continue
    meta = project.transect_metadata(transect)
    for name in names:
        s = pstore.get_oseries(name).squeeze().dropna()
        rows.append({
            "transect": transect,
            "series": name,
            "kind": "river" if name == meta["loc_river"] else "piezometer",
            "start": s.index.min(),
            "end": s.index.max(),
            "values": len(s),
            "time_step_min": s.index.to_series().diff().median().total_seconds() / 60,
            "min": s.min(),
            "max": s.max(),
        })

    folder = project.transect_dir(transect, "timeseries")
    folder.mkdir(parents=True, exist_ok=True)
    title = f"{project.name} - {transect}"
    f, ax = tlt.plot_select(pstore, title=transect, subset="all series", figsize=(10, 6), units=units)
    tlt.finalize_and_save(f, folder, title, tag="series_all", file_label=transect)
    ax = pstore.plots.data_availability("oseries", set_yticks=True, figsize=(10, 4))
    tlt.finalize_and_save(ax.figure, folder, title, tag="data_availability", file_label=transect)
    try:
        ax = pstore.maps.oseries(labels=True, s=60, figsize=(8, 8))
        tlt.finalize_and_save(ax.figure, folder, title, tag="series_map", file_label=transect)
    except Exception as e:
        print(f"{transect}: no map ({type(e).__name__}: {e})")

overview = pd.DataFrame(rows).set_index(["transect", "series"])
overview.style.format(precision=2)

## 2. Detail: one transect in a chosen period

In [ ]:
transect_name = transects[0]
meta = project.transect_metadata(transect_name)
xlim = [meta["tmin"], meta["tmax"]] if meta["tmin"] is not None else None  # or e.g. [pd.Timestamp("2025-11-01"), pd.Timestamp("2025-11-15")]

pstore = project.open_pastastore(transect_name)
tlt.plot_select(pstore, xlim=xlim, title=transect_name, subset="model period", figsize=(10, 6), units=units);